# Fine-tune Stable Diffusion for a Given Concept with DreamBooth

Original: https://github.com/fastai/diffusion-nbs/

Adapted: Antonio Esteves, UMinho

In this notebook, we will **personalise a Stable Diffusion model by fine-tuning it on a handful of images about a selected subject.** To do so, we will use a technique called [_DreamBooth_](https://arxiv.org/abs/2208.12242), which allows one to implant a subject (for exampe, a dog breed) into the domain of a pretrained model such that it can be synthesize that subject using a _unique identifier_ in the prompt.

## Prerequisites

Before diving into this notebook, one should read the:

* [The Hugging Face diffusion models class, unit 3 README](https://github.com/huggingface/diffusion-models-class/blob/main/unit3/README.md), which  contains a deep dive into Stable Diffusion
* DreamBooth [blog post](https://dreambooth.github.io/) to get a sense of what is possible to do with this technique
* Hugging Face [blog post](https://huggingface.co/blog/dreambooth) on best practices for fine-tuning Stable Diffusion with DreamBooth.

🚨 **Note:** the code in **this notebook requires at least 14GB of GPU VRAM** and is a simplified version of the [official training script](https://github.com/huggingface/diffusers/tree/main/examples/dreambooth) provided in `diffusers` library. It produces decent models for most applications, but we recommend experimenting with the advanced features like class preservation loss an fine-tuning the text encoder if we have at least 24GB of VRAM available. Check out the `diffusers` [documentation](https://huggingface.co/docs/diffusers/training/dreambooth) for more details.

## What is DreamBooth?

DreamBooth is a technique to teach new concepts to Stable Diffusion using a specialized form of fine-tuning. For example, people use this technique to create avatars of themselves.

The way DreamBooth works is as follows:

* Collect around 10-20 input images of a subject (e.g., a specific dog breed) and define a unique identifier [MyID] that refers to the subject. This identifier is usually an uncommon word like `mybreedofdog` which is implanted in different text prompts at inference time to place the subject in different contexts.
* Fine-tune the diffusion model by providing the images together with a text prompt like "A photo of a [mybreedofdog] dog" that contains the unique identifier and class name ("dog", in this example).
* (Optionally) Apply a special _class-specific prior preservation loss_, which leverages the semantic prior that the model has on the class and encourages it to generate diverse instances belong to the subject's class by injecting the class name in the text prompt. In practice, this step is only really needed for human faces.

An overview of the DreamBooth technique is shown in the image below.

![](https://dreambooth.github.io/DreamBooth_files/high_level.png)

### What can DreamBooth do?

Besides putting your subject in interesting locations, DreamBooth can be used for _**text-guided view synthesis**_, where the subject is viewed from different viewpoints as shown in the example below.

![](https://dreambooth.github.io/DreamBooth_files/novel_views.png)

DreamBooth can also be used to modify properties of the subject, such as colour or mixing up animal species!

![](https://dreambooth.github.io/DreamBooth_files/property_modification.png)

Now that we know some of the cool things DreamBooth can do, let us start training our own models!

## Step 1: Setup

If we are running the notebook on Google Colab or Kaggle, run the cell below to install the required libraries.

In [ ]:
%pip uninstall -y torch torchvision torchaudio transformers accelerate datasets diffusers bitsandbytes ftfy peft
%pip install   -qqU peft==0.18.1 diffusers transformers bitsandbytes accelerate ftfy datasets torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu126

To be able to push our model to the Hub, there are a few more steps to follow. First we have to create an [access token](https://huggingface.co/docs/hub/security-tokens) with _**write access**_ from our Hugging Face account and then execute the following cell and input the token.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

The final step is to install Git LFS.

In [ ]:
%%capture
!sudo apt -qq install git-lfs
!git config --global credential.helper store

## Step 2: Create an image dataset and upload it to the Hub

The next step is to **create a dataset of images about the selected subject** and upload it to the Hugging Face Hub.

* It is necessary around **10-20 images of the subject** that we wish to implant in the model. These can be photos we have taken or downloaded from platforms like [Unsplash](https://unsplash.com/). Alternatively, we can take a look at any of the [image datasets](https://huggingface.co/datasets?task_categories=task_categories:image-classification&sort=downloads) on the Hugging Face Hub for inspiration.
* For best results, we recommend using images of our subject from **different angles and perspectives**.

Once we have collected the images in a folder, we can upload them to the Hub by using the UI to drag and drop the  images. See [this guide](https://huggingface.co/docs/datasets/upload_dataset#upload-with-the-hub-ui) for more details, or watch the video below.

In [ ]:
'''
from IPython.display import YouTubeVideo

YouTubeVideo("HaN6qCr_Afc")
'''

Alternatively, we can load the dataset locally using the `imagefolder` feature of Hugging Face Datasets and then push it to the Hub.

```python
from datasets import load_dataset

dataset = load_dataset("imagefolder", data_dir="our_folder_with_images")
# Push to Hub
dataset.push_to_hub("dreambooth-tutorial-images")
dataset = dataset['train']
```

Once we have created the dataset, we can download it by using the `load_dataset()` function as follows.

In [ ]:
from datasets import load_dataset

#dataset_id = "lewtun/corgi"              # CHANGE THIS TO THE SELECTED {hub_username}/{dataset_id}
dataset_id = "ajesteves/caoserraestrela" # CHANGE THIS TO THE SELECTED {hub_username}/{dataset_id}
dataset    = load_dataset(dataset_id, split="train")

dataset

Now that we have our dataset, let us define a helper function to view a few of the images.

In [ ]:
from PIL import Image

def image_grid(imgs, rows, cols):
    assert len(imgs) == rows * cols
    w, h           = imgs[0].size
    grid           = Image.new("RGB", size=(cols * w, rows * h))
    grid_w, grid_h = grid.size
    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid


num_samples = 4
image_grid(dataset["image"][:num_samples], rows=1, cols=num_samples)

If this looks good, we can move onto the next step, creating a PyTorch dataset for training with DreamBooth.

## Step 3: Create a training dataset

To create a training set for our images we need a few components:

* An _instance prompt_ that is used to prime the model at the start of training. In most cases, using "a photo of [identifier] [class noun]" works quite well, e.g., "a photo of ccorgi dog" for our cute Corgi pictures.
    * **Note:** it is recommended to pick a unique / made up word like `mycorgi` to describe our subject. This will ensure that a common word already in the model's vocabulary is not overwritten.
* A _tokenizer_ to convert the instance prompt into input IDs that can be fed to the text encoder of Stable Diffusion.
* A set of _image transforms_, notably resizing the images to a common shape and normalizing the pixel values to a common mean and standard distribution.

With this in mind, let us start by defining the instance prompt.

In [ ]:
name_of_our_concept        = "csestrela"         # CHANGE THIS ACCORDING TO THE SELECTED SUBJECT
description_of_our_concept = "serra da estrela"  # CHANGE THIS ACCORDING TO THE SELECTED SUBJECT
tag_of_our_concept         = "SerraDaEstrela"    # CHANGE THIS ACCORDING TO THE SELECTED SUBJECT
type_of_thing              = "dog"               # CHANGE THIS ACCORDING TO THE CLASS OF SELECTED SUBJECT
instance_prompt            = f"a photo of {name_of_our_concept} {type_of_thing}"
print(f"Instance prompt: {instance_prompt}")

Next, we need to create a PyTorch `Dataset` object that implements the `__len__` and `__getitem__` mandatory methods.

In [ ]:
from torch.utils.data import Dataset
from torchvision      import transforms

class DreamBoothDataset(Dataset):
    def __init__(self, dataset, instance_prompt, tokenizer, size=512):
        self.dataset         = dataset
        self.instance_prompt = instance_prompt
        self.tokenizer       = tokenizer
        self.size            = size
        self.transforms      = transforms.Compose(
            [
                transforms.Resize(size),
                transforms.CenterCrop(size),
                transforms.ToTensor(),
                transforms.Normalize([0.5], [0.5]),
            ]
        )

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        example = {}
        image   = self.dataset[index]["image"]
        example["instance_images"]     = self.transforms(image)
        example["instance_prompt_ids"] = self.tokenizer(
            self.instance_prompt,
            padding    = "do_not_pad",
            truncation = True,
            max_length = self.tokenizer.model_max_length,
        ).input_ids
        return example

Let us now check that this works by loading the CLIP tokenizer associated with the text encoder of the original Stable Diffusion model, and then creating the training dataset.

In [ ]:
from transformers import CLIPTokenizer

# The Stable Diffusion model that we will fine-tune
model_id  = "CompVis/stable-diffusion-v1-4"
tokenizer = CLIPTokenizer.from_pretrained(
    model_id,
    subfolder = "tokenizer",
)

# Create the Dreambooth custom dataset
train_dataset = DreamBoothDataset(
  dataset,
  instance_prompt,
  tokenizer
)
# Get the first sample from the dataset
train_dataset[0]

## Step 4: Define a data collator

Now that we have a training dataset, the next thing we need is to define a _data collator_. A data collator is a function that collects elements in a batch of data and applies some logic to form a single tensor we can provide to the model. To learn more about collators, one can check out the following video from the [Hugging Face Course](hf.co/course).

In [ ]:
'''
YouTubeVideo("-RPeakdlHYo")
'''

For DreamBooth, our data collator needs to provide the model with the prompt's token IDs from the tokenizer and the pixel values from the images as a stacked tensor. The function below does the job.

In [ ]:
import torch

def collate_fn(examples):
    input_ids    = [example["instance_prompt_ids"] for example in examples]
    pixel_values = [example["instance_images"] for example in examples]
    pixel_values = torch.stack(pixel_values)
    pixel_values = pixel_values.to(memory_format=torch.contiguous_format).float()

    input_ids = tokenizer.pad(
        {"input_ids": input_ids}, padding=True, return_tensors="pt"
    ).input_ids

    batch = {
        "input_ids":    input_ids,
        "pixel_values": pixel_values,
    }
    return batch

## Step 5: Load the components of the Stable Diffusion pipeline

We nearly have all the pieces ready for training! As we saw in the notebook about Stable Diffusion, the pipeline is composed of several models:

* A text encoder that converts the prompts into text embeddings. Here we will use CLIP since it is the encoder that was used to train Stable Diffusion v1.4.
* A variational autoencoder (VAE)that converts the images to compressed representations (i.e., latents) and decompresses them at inference time.
* A UNet that applies the denoising operation on the latents obtained from the VAE.

We can load all these components using the Hugging Face `diffusers` and `transformers` libraries as follows.

In [ ]:
from diffusers    import AutoencoderKL, UNet2DConditionModel
from transformers import CLIPImageProcessor, CLIPTextModel

text_encoder      = CLIPTextModel.from_pretrained(
  model_id,
  subfolder="text_encoder"
)
vae               = AutoencoderKL.from_pretrained(
  model_id,
  subfolder="vae"
)
unet              = UNet2DConditionModel.from_pretrained(
  model_id,
  subfolder="unet"
)
feature_extractor = CLIPImageProcessor.from_pretrained(
  "openai/clip-vit-base-patch32"
)

## Step 6: Fine-tune the model

Finally, we are going to train our model with DreamBooth. As shown in the [Hugging Face's blog post](https://huggingface.co/blog/dreambooth), the most essential hyperparameters to tweak are the learning rate and the number of training steps.

In general, we will get better results with a lower learning rate at the expense of needing to increase the number of training steps. The values below are a good starting point, but it may be necessary to adjust them according to our dataset.

In [ ]:
learning_rate   = 2e-06
max_train_steps = 8000

Next, let us gather the other hyperparameters we need in a `Namespace` object to make it easier to configure the training run.

In [ ]:
from argparse import Namespace

args = Namespace(
    pretrained_model_name_or_path = model_id,
    resolution                    = 512,           # Reduce this to save some memory
    train_dataset                 = train_dataset,
    instance_prompt               = instance_prompt,
    learning_rate                 = learning_rate,
    max_train_steps               = max_train_steps,
    train_batch_size              = 1,
    gradient_accumulation_steps   = 1,      # Increase this to lower memory usage
    max_grad_norm                 = 1.0,
    gradient_checkpointing        = True,   # Set this to True to lower the memory usage
    use_8bit_adam                 = True,   # Use 8bit optimizer from bitsandbytes
    seed                          = 3434554,
    sample_batch_size             = 2,
    output_dir                    = "sd-dreambooth-tunned", # Where to save the pipeline
)

The final step is to define a `train()` function that wraps the training logic and can be passed to the `accelerate` library to handle training on one or more GPUs. One can watch the next video about `accelerate`.

In [ ]:
'''
YouTubeVideo("s7dy8QRgjJ0")
'''

In [ ]:
import math
import torch.nn.functional as     F
from   accelerate          import Accelerator
from   accelerate.utils    import set_seed
from   diffusers           import DDPMScheduler, PNDMScheduler, StableDiffusionPipeline
from   diffusers.pipelines.stable_diffusion import StableDiffusionSafetyChecker
from   torch.utils.data    import DataLoader
from   tqdm.auto           import tqdm

def train(text_encoder, vae, unet):

    accelerator = Accelerator(
        gradient_accumulation_steps=args.gradient_accumulation_steps,
    )

    set_seed(args.seed)

    if args.gradient_checkpointing:
        unet.enable_gradient_checkpointing()

    # Use 8-bit Adam for lower memory usage or to fine-tune the model in 16GB GPUs
    if args.use_8bit_adam:
        import bitsandbytes as bnb
        optimizer_class = bnb.optim.AdamW8bit
    else:
        optimizer_class = torch.optim.AdamW

    optimizer = optimizer_class(
        unet.parameters(),       # Only optimize the UNet
        lr = args.learning_rate,
    )

    noise_scheduler = DDPMScheduler(
        beta_start          = 0.00085,
        beta_end            = 0.012,
        beta_schedule       = "scaled_linear",
        num_train_timesteps = 1000,
    )

    train_dataloader = DataLoader(
        args.train_dataset,
        batch_size = args.train_batch_size,
        shuffle    = True,
        collate_fn = collate_fn,
    )

    unet, optimizer, train_dataloader = accelerator.prepare(
        unet,
        optimizer,
        train_dataloader
    )

    # Move 'text_encode' and 'vae' to GPU
    text_encoder.to(accelerator.device)
    vae.to(accelerator.device)

    # We need to recalculate our total training steps as
    # the size of the training dataloader may have changed
    num_update_steps_per_epoch = math.ceil(
        len(train_dataloader) / args.gradient_accumulation_steps
    )
    num_train_epochs = math.ceil(args.max_train_steps / num_update_steps_per_epoch)

    # Train the UNet
    total_batch_size = (
        args.train_batch_size
        * accelerator.num_processes
        * args.gradient_accumulation_steps
    )
    # Only show the progress bar once on each machine
    progress_bar = tqdm(
        range(args.max_train_steps), disable=not accelerator.is_local_main_process
    )
    progress_bar.set_description("Steps")
    global_step = 0

    for epoch in range(num_train_epochs):
        unet.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(unet):
                # Convert the images to latent tensors
                with torch.no_grad():
                    latents = vae.encode(batch["pixel_values"]).latent_dist.sample()
                    latents = latents * 0.18215

                # Sample noise that we will add to the latents
                noise = torch.randn(latents.shape).to(latents.device)
                bsz   = latents.shape[0] # batch size
                # Sample a random timestep for each image in the batch
                timesteps = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (bsz,),
                    device=latents.device,
                ).long()

                # Add noise to each latent according to the noise level associated
                # to the corresponding timestep (this is the forward diffusion process)
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Get the conditioning text embeddings
                with torch.no_grad():
                    encoder_hidden_states = text_encoder(batch["input_ids"])[0]

                # Predict the noise to remove
                noise_pred = unet(
                    noisy_latents, timesteps, encoder_hidden_states
                ).sample

                # claculate the MSE between the predicted and added noises
                loss = (
                    F.mse_loss(noise_pred, noise, reduction="none")
                    .mean([1, 2, 3])
                    .mean()
                )

                # Calculate loss gradients relative to the model's weights
                accelerator.backward(loss)

                # Clip the gradient's norm if necessary
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), args.max_grad_norm)

                # Update the model's weights using the gradients
                optimizer.step()

                # Reset the accumulated gradients
                optimizer.zero_grad()

            # Checks if the accelerator has performed an optimization step behind the scenes
            if accelerator.sync_gradients:
                progress_bar.update(1)
                global_step += 1

            logs = {"loss": loss.detach().item()}
            progress_bar.set_postfix(**logs)

            if global_step >= args.max_train_steps:
                break

        accelerator.wait_for_everyone()

    # Create the pipeline using the trained modules and save it
    if accelerator.is_main_process:
        print(f"Loading pipeline and saving it to {args.output_dir} ...")
        scheduler = PNDMScheduler(
            beta_start     = 0.00085,
            beta_end       = 0.012,
            beta_schedule  = "scaled_linear",
            skip_prk_steps = True,
            steps_offset   = 1,
        )

        pipeline = StableDiffusionPipeline(
            text_encoder   = text_encoder,
            vae            = vae,
            unet           = accelerator.unwrap_model(unet),
            tokenizer      = tokenizer,
            scheduler      = scheduler,
            safety_checker = StableDiffusionSafetyChecker.from_pretrained(
              "CompVis/stable-diffusion-safety-checker"
            ),
            feature_extractor = feature_extractor,
        )

        pipeline.save_pretrained(args.output_dir)

Now that we have the training function defined, let us train the UNet. Depending on the size of our dataset and type of GPU, this can take anywhere from 5 minutes to 1 hour to run.

In [ ]:
from accelerate import notebook_launcher

num_of_gpus = 1  # CHANGE THIS TO MATCH THE NUMBER OF GPUS WE HAVE
notebook_launcher(
  train,
  args          = (text_encoder, vae, unet),
  num_processes = num_of_gpus
)

When running on a single GPU, we can free up some memory for the next section by copying the code below into a new cell and running it. For multi-GPU machines, `accelerate` does not allow _any_ cell to directly access the GPU with `torch.cuda`, so we do not recommend using this trick in those cases.

```python
with torch.no_grad():
    torch.cuda.empty_cache()
```

## Step 7: Run inference and inspect generations

Now, that we have trained the model, let us generate some images with it to see how it behaves. First, we will load the pipeline from the output directory where we saved the model.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    args.output_dir,
    torch_dtype = torch.float16,
).to("cuda")

Next, let us generate a few images. The `prompt` variable will later be used to set the default on the Hugging Face Hub widget, so experiment a bit to find a good one. One might also want to try creating elaborate prompts with [CLIP Interrogator](https://huggingface.co/spaces/pharma/CLIP-Interrogator).

In [ ]:
# Pick a funny prompt here and it will be used as the widget's default
# when we push to the Hub in the next section
prompt = f"a photo of {name_of_our_concept} {type_of_thing} close to Eiffel"

# Tune the guidance to control how closely the generations follow the prompt.
# Values in the range 7-11 usually work best.
guidance_scale = 8

num_cols   = 2
all_images = []
for _ in range(num_cols):
    images = pipe(prompt, guidance_scale=guidance_scale).images
    all_images.extend(images)

image_grid(all_images, 1, num_cols)

## Step 8: Push the tuned model to the Hub

An optional final step can be to push the fine-tuned model to the Hugging Face Hub.

First, we need to define a name for our model repo. By default, we use the unique identifier and class name.

In [ ]:
# Create a name for our model on the HF Hub. Spaces are NOT allowed.
model_name = f"{name_of_our_concept}-{type_of_thing}"

Next, add a brief description on the type of model we trained or any other information that we want to share.

In [ ]:
# Describes the model we trained and the concept we implanted on it 
description = f"""
This is a Stable Diffusion model fine-tuned on images of {type_of_thing}'s {description_of_our_concept}.
"""

Finally, run the cell below to create a repo on the Hub and push all the files with a nice model card.

In [ ]:
# Code to upload a pipeline saved locally to the HF hub
from huggingface_hub import HfApi, ModelCard, create_repo, get_full_repo_name

# Set up repo and upload files
hub_model_id = get_full_repo_name(model_name)
create_repo(hub_model_id)
api          = HfApi()
api.upload_folder(folder_path=args.output_dir, path_in_repo="", repo_id=hub_model_id)

content = f"""
---
license: creativeml-openrail-m
tags:
- pytorch
- diffusers
- stable-diffusion
- text-to-image
- diffusion-models-class
- dreambooth
- {type_of_thing}
- {tag_of_our_concept}
widget:
- text: {prompt}
---

# DreamBooth model for the {name_of_our_concept} concept trained by {api.whoami()["name"]} on the {dataset_id} dataset.

This is a Stable Diffusion model fine-tuned on the {name_of_our_concept} concept with DreamBooth.
It can be used by modifying the `instance_prompt`: **{instance_prompt}**

## Description

{description}

## Usage

```python
from diffusers import StableDiffusionPipeline

pipeline = StableDiffusionPipeline.from_pretrained('{hub_model_id}')
image    = pipeline().images[0]
image
```
"""

card    = ModelCard(content)
hub_url = card.push_to_hub(hub_model_id)
print(f"Upload successful! Model can be found here: {hub_url}")